# GeoEnrichment User Guide

## What is GeoEnrichment

GeoEnrichment provides the ability to get facts about a location or area. Using GeoEnrichment, you can get information about the people, places, and businesses in a specific area or within a certain distance or drive time from a location. More specifically, by submitting a point or polygon to the GeoEnrichment service, you can retrieve the demographics and other relevant characteristics associated with the surrounding area. You can also use the GeoEnrichment service to obtain additional geographic context (for example, the ZIP Code of a location) and geographic boundaries (for example, the geometry for a drive-time service area).

This service enables you to answer questions about locations that you can't answer with maps alone. For example: What kind of people live here? What do people like to do in this area? What are their habits and lifestyles? What kind of businesses are in this area?

Site analysis is a popular application of this type of data enrichment. For example, the GeoEnrichment service can be leveraged to study the population that would be affected by the development of a new community center within their neighborhood. With the service, the proposed site can be submitted, and the demographics and other relevant characteristics associated with the area around the site will be returned.

### What is Required to Use GeoEnrichment?

GeoEnrichment cannot be used with anonymous users, users must be logged into a GIS with GeoEnrichment enabled.  The service uses credits, therefore a user must be able to consume credits to generate reports and enrich data.

### Benefits of the Python API

The Python API returns the results as either a Spatial or Pandas' DataFrame.  This means the data is easy to both visualize and work with.  The results can be saved to disk as featureclasses or to Feature Layers on the site's GIS.

## Getting Started

A user must be logged in to a GIS in order to use  the GeoEnrichment service.  A user must have credits on the GIS and have the GeoEnrichment enabled on the organization. 

In [1]:
from arcgis.gis import GIS
import getpass
username = getpass.getpass()
password = getpass.getpass()
gis = GIS(username=username, password=password, verify_cert=False)

········
········


### Enrich Data

Enrich data method adds information to a geometry object.  The result of the enrichment, if in a valid location, is a dataset that has fields/attributes that come from other lifestyle/demographic datasets.

#### Forms of Locations

- **Input XY locations** - The most common method to determine the center point for a study area is a set of one or many point locations defined as XY locations. More specifically, one or many input points (latitude and longitude) can be provided to the service to set the study areas that you want to enrich with additional information. You can create a buffer ring or drive-time service area around the points to aggregate data for the study areas. You can also return enrichment data for buffers around input line features.

    + **Example XY Location: ** {"geometry":{"x":-122.435,"y":37.785},"attributes":{"id":"1"}}
    + **Example Buffered Location: ** {"geometry":{"x": -122.435, "y": 37.785},"areaType": "NetworkServiceArea","bufferUnits": "Hours","bufferRadii": [1],"travel_mode":"Driving"}

- **Network service areas** - The GeoEnrichment service allows you to create drive time service areas around points as well as other advanced service areas such as walking and trucking. Using the enrich method, you can define the network service area properties in the studyAreas and studyAreasOptions parameters for input point locations such as XY coordinates and addresses.

    + Driving—This is the default value. It models the movement of cars and other similar small automobiles, such as pickup trucks. Travel obeys one-way roads, avoids illegal turns, and follows other rules that are specific to cars. Dynamic travel speeds based on traffic are used where it is available when you specify a start time (time_of_day).
    + Trucking—Models basic truck travel by preferring designated truck routes and using typical truck speeds. Routes must obey one-way roads, avoid illegal turns, and so on. (To model the characteristics of a specific truck, such as its height and weight, choose the Custom travel mode instead.)
    + Walking—Follows paths and roads that allow pedestrian traffic. (Choose the Custom travel mode instead if you want to customize the walking mode by, for example, setting the walking speed to a value other than the default 5 kilometers per hour.)
    + **Example: ** [{"geometry":{"x":-122.435,"y":37.785},"areaType": "NetworkServiceArea","bufferUnits": "Minutes","bufferRadii": [10],"travel_mode":"Driving"},{"geometry":{"x":-117.1956,"y":34.0572},"areaType": "NetworkServiceArea","bufferUnits": "Minutes","bufferRadii": [30],"travel_mode":"Walking"}]
- **Named statistical areas** - In all previous examples of different study area types, locations were defined as either points or polygons. Study area locations can also be passed as one or many named statistical areas. This form of study area lets you define an area by the ID of a standard geographic statistical feature, such as a census or postal area, for example, to obtain enrichment information for a U.S. state, county, or ZIP Code or a Canadian province or postal code.
    + **Example:** [{"sourceCountry":"US","layer":"US.ZIP5","ids":["92373","92129"]}]
- **Input Polygons** - Locations can given as polygon geometries.
    + [{"geometry":{"rings":[[[-117.185412,34.063170],[-122.81,37.81],[-117.200570,34.057196],[-117.185412,34.063170]]],"spatialReference":{"wkid":4326}},"attributes":{"id":"1","name":"optional polygon area name"}}]
- **Street address locations** - Locations can be passed as input street addresses. 
    + **Example:** "380 New York St, Redlands, CA"

In [2]:
from arcgis import geoenrichment as ge

In [3]:
df = ge.enrich(study_areas=[
                       {"geometry":{"x":-122.435,"y":37.785},"attributes":{"id":"1"}},
                       {"geometry":{"x":-122.433,"y":37.734},"attributes":{"id":"2"}},
                       {"sourceCountry":"US","layer":"US.ZIP5","ids":["92373","92129"]},
                       {"geometry":{"x": -122.435, "y": 37.785},"areaType": "NetworkServiceArea",
                        "bufferUnits": "Hours","bufferRadii": [1],"travel_mode":"Driving"},
                       {"address":{"text":"12 Concorde Place Toronto ON M3C 3R8","sourceCountry":"Canada"}},
                       {"address":{"text":"380 New York St Redlands CA 92373","sourceCountry":"US"}},
                       {"geometry":{"rings":[[[-117.185412,34.063170],[-122.81,37.81],
                                              [-117.200570,34.057196],[-117.185412,34.063170]]],
                                    "spatialReference":{"wkid":4326}},
                        "attributes":{"id":"3","name":"optional polygon area name"}}])
df.head()

,AVGHHSZ,AVGHHSZ_I,HasData,ID_0,OBJECTID,StdGeographyID,StdGeographyLevel,StdGeographyName,TOTFEMALES,TOTFEMALES_P,...,Y,aggregationMethod,areaType,bufferRadii,bufferUnits,bufferUnitsAlias,id,name,sourceCountry,SHAPE
0,1.80,70,1,0,1,NaN,NaN,NaN,53872,50.44,...,NaN,BlockApportionment:US.BlockGroups,RingBuffer,1.0,esriMiles,Miles,1,NaN,US,"{'rings': [[[-122.43499999999999, 37.799499596..."
1,2.76,107,1,1,2,NaN,NaN,NaN,33983,50.08,...,NaN,BlockApportionment:US.BlockGroups,RingBuffer,1.0,esriMiles,Miles,2,NaN,US,"{'rings': [[[-122.43299999999999, 37.748499722..."
2,2.40,93,1,NaN,3,92373,US.ZIP5,Redlands,17522,52.35,...,NaN,Query:US.ZIP5,NaN,NaN,NaN,NaN,2,NaN,US,"{'rings': [[[-117.21461999975689, 34.065140000..."
3,3.03,117,1,NaN,4,92129,US.ZIP5,San Diego,27449,50.42,...,NaN,Query:US.ZIP5,NaN,NaN,NaN,NaN,2,NaN,US,"{'rings': [[[-117.12766999997676, 33.000260001..."
4,2.56,99,1,NaN,5,NaN,NaN,NaN,2170932,50.60,...,NaN,BlockApportionment:US.BlockGroups,NetworkServiceArea,1.0,Hours,Drive Time Hours,3,NaN,US,"{'rings': [[[-122.0758972167445, 37.3065509802..."


The Enrich method returns a **Spatial DataFrame** can can either be used for mapping via the **gis.content.import_data** method or for local analysis.  If return_geometry is set to false, a Pandas' DataFrame is returned instead.  


### Viewing Available Reports

To look up what reports are available for a given country, the find report operation returns a list of report ids along with metadata and exportable formats.

#### Example:Find Swedish Available Reports

In [4]:
reports = ge.find_report(country="SE")
reports.head()

,formats,headers,metadata,reportID
0,[pdf],"[locationname, address, latitude, longitude, a...","{'name': 'Site Map Report', 'boundaryVintage':...",site_map
1,"[pdf, xlsx]","[locationname, address, latitude, longitude, a...","{'name': 'Sweden Summary Report', 'boundaryVin...",SwedenSummary


#### Example: Get Swedish Reports Metadata


In [5]:
metadata = ge.report_metadata('SE')
metadata

,author,boundaryVintage,boundaryVintageDescription,categories,countries,coverage,creationDate,dataVintage,dataVintageDescription,dataset,hierarchy,keywords,lastRevisionDate,name,reportID,title,type
0,Esri,2010,Data displayed and aggregated on these reports...,[Maps],None,"North America, Europe, and the rest of the world",1355122800000,N/A,N/A,,None,Site Map,1355468400000,Site Map Report,site_map,Site Map,esriReportTemplateMapReport
1,Esri,2015,2015,[Summary Reports],SE,SE,1472713200000,"2014,2015",This report contains data from Michael Bauer R...,SWE_MBR_2015,census,"MBR, Demographic, Data",1472713200000,Sweden Summary Report,SwedenSummary,Sweden Summary Report,esriReportTemplateMultiColumn


### Data Collections

The GeoEnrichment service uses the concept of a data collection to define the data attributes returned by the enrichment service. Each data collection has a unique name that acts as an ID that is passed in the dataCollections parameter of the GeoEnrichment service.

Some data collections (such as default) can be used in all supported countries. Other data collections may only be available in one or a collection of countries. Data collections may only be available in a subset of countries because of differences in the demographic data that is available for each country. 

In [6]:
df = ge.data_collections(country="CA", 
                         dataset="EducationalAttainment", 
                         variables=["percent"])
df.head()

,alias,derivative,description,fieldCategory,filteringTags,hideInDataBrowser,id,percentBase,percentBaseAlias,popularity,precision,provider,type,units,vintage
0,2012 Hh Pop 15+: Education,False,2012 Household Population 15+ For Educational ...,2012 Household Population 15 Years or Over by ...,"[{'name': 'Dataset', 'id': 'Dataset', 'value':...",False,EHYEDUHP15,NaN,NaN,NaN,0,Environics,esriFieldTypeDouble,count,2012
1,"2012 Pop 15: No Cert, Dip/Deg",False,"2012 Pop 15: No Certificate, Diploma Or Degree",2012 Household Population 15 Years or Over by ...,"[{'name': 'Dataset', 'id': 'Dataset', 'value':...",False,EHYEDUNCDD,EHYEDUHP15,2012 Hh Pop 15+: Education,NaN,0,Environics,esriFieldTypeDouble,count,2012
2,"2012 Pop 15: No Cert, Dip/Deg: Percent",True,"2012 Pop 15: No Certificate, Diploma Or Degree...",2012 Household Population 15 Years or Over by ...,"[{'name': 'Dataset', 'id': 'Dataset', 'value':...",False,EHYEDUNCDD_P,NaN,NaN,NaN,2,NaN,esriFieldTypeDouble,pct,2012
3,2012 Pop 15: High Sch/Equiv,False,2012 Pop 15: High School Certificate or Equiva...,2012 Household Population 15 Years or Over by ...,"[{'name': 'Dataset', 'id': 'Dataset', 'value':...",False,EHYEDUHSCE,EHYEDUHP15,2012 Hh Pop 15+: Education,NaN,0,Environics,esriFieldTypeDouble,count,2012
4,2012 Pop 15: High Sch/Equiv: Percent,True,2012 Pop 15: High School Certificate or Equiva...,2012 Household Population 15 Years or Over by ...,"[{'name': 'Dataset', 'id': 'Dataset', 'value':...",False,EHYEDUHSCE_P,NaN,NaN,NaN,2,NaN,esriFieldTypeDouble,pct,2012


### Standard geography query

The GeoEnrichment service provides a helper method that returns standard geography IDs and features for the supported geographic levels in the United States and Canada.

As indicated throughout this documentation guide, the GeoEnrichment service uses the concept of a study area to define the location of the point or area that you want to enrich with additional information. Locations can also be passed as one or many named statistical areas. This form of a study area lets you define an area by the ID of a standard geographic statistical feature, such as a census or postal area. For example, to obtain enrichment information for a U.S. state, county or ZIP Code or a Canadian province or postal code, the Standard Geography Query helper method allows you to search and query standard geography areas so that they can be used in the GeoEnrichment method to obtain facts about the location.

The most common workflow for this service is to find a FIPS (standard geography ID) for a geographic name. For example, you can use this service to find the FIPS for the county of San Diego which is 06073. You can then use this FIPS ID within the GeoEnrichment service study area definition to get geometry and optional demographic data for the county. This study area definition is passed as a parameter to the GeoEnrichment service to return data defined in the enrichment pack and optionally return geometry for the feature.


In [7]:
df = ge.standard_geography_query(source_country='US',
                            layers=['US.States'],
                            ids=['06'],
                            return_geometry=True)
df

,AreaID,AreaName,CountryAbbr,DataLayerID,DatasetID,MajorSubdivisionAbbr,MajorSubdivisionName,MajorSubdivisionType,ObjectId,Score,SHAPE
0,06,California,US,US.States,USA_ESRI_2017,CA,California,State,1,100,"{'rings': [[[-122.26220299985978, 42.007666999..."


## Select businesses

The select_businesses method returns business points matching a given search criteria. Business points can be selected using any combination of three search criteria: search string, spatial filter and business type. A business point will be selected if it matches all search criteria specified.

In [8]:
df = ge.select_businesses(search_string="Mechanics",
                          return_geometry=True,
                          spatial_filter={"Locations":["NY,TONAWANDA,14150",
                                                       "NJ, CAMDEN, 08102",
                                                       "KY,LOUISVILLE,40204",
                                                       "WA,SEATTLE,98108"]})
df.head()

,ADDR,CITY,CONAME,EMPNUM,FRNCOD,HDBRCH,ISCODE,LOCNUM,LOC_NAME,NAICS,...,SIC,SOURCE,SQFTCODE,STATE,STATE_NAME,STATUS,STREET,ZIP,ZIP4,SHAPE
0,50 JAMES ST,TONAWANDA,LAKESIDE MECHANICAL,2,,,,654553072,PointAddress,23821038,...,171114,INFOGROUP,A,NY,New York,M,JAMES ST,14150,3806,"{'x': -78.8853000002279, 'spatialReference': {..."
1,95 PIRSON PKWY,TONAWANDA,M J MECHANICAL SVC INC,228,,3,,864029533,PostalExt,23821038,...,171114,INFOGROUP,D,NY,New York,M,PIRSON PKWY,14150,6776,"{'x': -78.8835000000791, 'spatialReference': {..."
2,433 MARKET ST,CAMDEN,WORTH & CO,10,,,,705776511,PointAddress,23821038,...,171114,INFOGROUP,B,NJ,New Jersey,M,MARKET ST,08102,1572,"{'x': -75.1218999997914, 'spatialReference': {..."
3,1346 NIAGARA FALLS BLVD,TONAWANDA,VALU AUTO CARE CTR,5,,,,117506162,PointAddress,81111817,...,753914,INFOGROUP,B,NY,New York,M,NIAGARA FALLS BLVD,14150,8917,"{'x': -78.8227000004404, 'spatialReference': {..."
4,1106 SHERIDAN DR,TONAWANDA,PARISE MECHANICAL INC,18,,,,578683799,PointAddress,23822020,...,171102,INFOGROUP,B,NY,New York,M,SHERIDAN DR,14150,8045,"{'x': -78.8934000004488, 'spatialReference': {..."


### Creating a Report


The Create Report method allows you to create many types of high quality reports for a variety of use cases describing the input area. If a point is used as a study area, the service will create a 1-mile ring buffer around the point to collect and append enrichment data. Optionally, you can create a buffer ring or drive-time service area around points of interest to generate PDF or Excel reports containing relevant information for the area on demographics, consumer spending, tapestry market, business or market potential.

Report options are available and can be used to describe and gain a better understanding about the market, customers/clients and competition associated with an area of interest.

In [13]:
study_area = [{"geometry":{"rings":[[[-117.26,32.81],[-117.40,32.92],[-117.12,32.80],[-117.26,32.81]]],
                      "spatialReference":{"wkid":4326}},"attributes":{"id":"Polygon 1","name":"Optional Name 1"}
             },
            {"address":{"text":"380 New York St. Redlands, CA 92373"}}]
import os
r = ge.create_report(study_areas=study_area,
                     report="census2010_profile",
                     export_format="PDF",
                     use_data={"sourceCountry":"US"},
                     out_folder=r"c:\temp", out_name="report.pdf")
print(r)

c:\temp\report.pdf


## Save the Results

The results can be saved back to a GIS or to a feature class.  

#### Example: Save Business Data to a Feature Layer

In [10]:
gis.content.import_data(df=df, title="Enriched Data Demo")

<Item title:"Enriched Data Demo" type:Feature Layer Collection owner:AndrewSolutions>

## Summary

This guide showed users how to enhance spatial data with rich demographic data. Once the demographic data is joined to the spatial data, the next step is perform addition analysis, map the data, or save it. 

The Spatial DataFrame or Pandas' DataFrame provides GIS users with the ability to manage and manipulate the data further.